# E337 | Big Data | Trabajo práctico nº1 | Otoño 2026

#### por Constanza María Efkhanian, Julián Fabrizio Notario y María Florencia Pascale

### Profesora: María Noelia Romero
### Tutora: Luciana Azul Ramirez


# Parte I: Familiarización con la base EPH y limpieza

#### 1.a. Elección de región

Para el presente trabajo y los próximos, elegimos la región del Gran Buenos Aires.

### Bibliotecas

In [55]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Directorio

In [56]:
os.chdir("/Users/juliannotario/Desktop/Big Data/Pruebas de código")


In [57]:
df_2005_raw = pd.read_stata('usu_individual_3105.dta', convert_categoricals=False, convert_missing=False) 
df_2025_raw = pd.read_excel('usu_individual_T325.xls')

In [58]:
df_2005_raw.head(4)
df_2025_raw.head(4)

,CODUSU,ANO4,TRIMESTRE,NRO_HOGAR,COMPONENTE,H15,REGION,MAS_500,AGLOMERADO,PONDERA,...,V21_03_M,V22_01_M,V22_02_M,V22_03_M,P_DECCF,P_RDECCF,P_GDECCF,P_PDECCF,P_IDECCF,P_ADECCF
0,TQRMNOTYPHKOKQCDEHNHB00867020,2025,3,1,8,0,42,S,27,528,...,0,0,0,0,1.0,1.0,1.0,NaN,1.0,1.0
1,TQRMNOTYPHKOKQCDEHNHB00867020,2025,3,1,9,0,42,S,27,528,...,0,0,0,0,1.0,1.0,1.0,NaN,1.0,1.0
2,TQRMNOTYPHKOKQCDEHNHB00867020,2025,3,1,1,1,42,S,27,528,...,0,0,0,0,1.0,1.0,1.0,NaN,1.0,1.0
3,TQRMNOTYPHKOKQCDEHNHB00867020,2025,3,1,2,1,42,S,27,528,...,0,0,0,0,1.0,1.0,1.0,NaN,1.0,1.0


In [59]:
# Hacemos lo siguiente para contar la cantidad de observaciones del Gran Buenos Aires en la base de datos del 2005: 

count = 0

for i in df_2005_raw["region"]:
    if i == "Gran Buenos Aires":
        count += 1
    
print(count)

# Lo mismo para el 2025: 
count2 = 0

for j in df_2025["REGION"]:
    if j == 1:
        count2 += 1
    
print(count2)


0
2709


### Verificamos como están compuestas las variables de los dataframes

In [60]:
# Pasamos todas las variables a mayúsculas para armonizarlas:
df_2005_raw.columns = df_2005_raw.columns.str.upper().str.strip()
df_2025_raw.columns = df_2025_raw.columns.str.upper().str.strip()

# Elegimos variables:
variables = ['ANO4','PONDERA','CH04','CH06','CH07','CH08','NIVEL_ED',
             'ESTADO','CAT_INAC','IPCF','CAT_OCUP','PP07H','P21','PP04A','PP04C']

# Tamaño de las variables
print("\n TAMAÑO DE CADA BASE (sin filtrar)")
print(f"  2005: {df_2005_raw.shape[0]:,} filas x {df_2005_raw.shape[1]} columnas")
print(f"  2025: {df_2025_raw.shape[0]:,} filas x {df_2025_raw.shape[1]} columnas")

# Tipos de datos dentro de las variables
print("\n TIPO DE DATO POR VARIABLE")
print(f"  {'Variable':<12}  {'dtype 2005':>12}  {'dtype 2025':>12}")
for v in variables:
    d05 = str(df_2005_raw[v].dtype) if v in df_2005_raw.columns else "AUSENTE"
    d25 = str(df_2025_raw[v].dtype) if v in df_2025_raw.columns else "AUSENTE"
    flag = "  <<< DISTINTOS" if d05 != d25 else ""
    print(f"  {v:<12}  {d05:>12}  {d25:>12}{flag}")


# Acá vemos cuántas N/As hay en las variables antes de hacer el merge. La idea es que después de hacer el merge nos volvamos a 
# fijar cuantas N/As hay, y si concuerdan, se hizo bien el merge.
print("\n % NaN POR VARIABLE (antes de limpiar)")
print(f"  {'Variable':<12}  {'NaN% 2005':>10}  {'NaN% 2025':>10}")
for v in variables:
    p05 = df_2005_raw[v].isnull().sum() / len(df_2005_raw) * 100
    p25 = df_2025_raw[v].isnull().sum() / len(df_2025_raw) * 100
    print(f"  {v:<12}  {p05:>9.1f}%  {p25:>9.1f}%")


 TAMAÑO DE CADA BASE (sin filtrar)
  2005: 47,647 filas x 176 columnas
  2025: 16,383 filas x 235 columnas

 TIPO DE DATO POR VARIABLE
  Variable        dtype 2005    dtype 2025
  ANO4               float64         int64  <<< DISTINTOS
  PONDERA            float64         int64  <<< DISTINTOS
  CH04               float64         int64  <<< DISTINTOS
  CH06               float64         int64  <<< DISTINTOS
  CH07               float64         int64  <<< DISTINTOS
  CH08               float64         int64  <<< DISTINTOS
  NIVEL_ED           float64         int64  <<< DISTINTOS
  ESTADO             float64         int64  <<< DISTINTOS
  CAT_INAC           float64         int64  <<< DISTINTOS
  IPCF               float64       float64
  CAT_OCUP           float64         int64  <<< DISTINTOS
  PP07H              float64       float64
  P21                float64         int64  <<< DISTINTOS
  PP04A              float64       float64
  PP04C              float64       float64

 % NaN POR

### Limpieza de los DataFrames

In [61]:
# Primero, filtramos sólo por las columnas que incluyen al Gran Buenos Aires (para ambos DataFrames):

# Para esta sección, utilizamos ayuda de la IA. 

df_2005_GBA = df_2005_raw[df_2005_raw["REGION"] == 1] # crea un nuevo DataFrame con solo las filas de GBA
print(df_2005_GBA.head(4))
df_2025_GBA = df_2025_raw[df_2025_raw["REGION"] == 1] # Crea nuevo dataframe solo con la región 1 (GBA)

# En lo que sigue, debemos unificar las bases de datos y seleccionar las variables de interés. 
## Recordar: deben ser 15 variables de interés. Entre ellas debemos encontrar: CH04, CH06, CH07, CH08, NIVEL ED, ESTADO, CAT_INAC, IPCF
        

     CODUSU  NRO_HOGAR  COMPONENTE  H15    ANO4  TRIMESTRE  REGION MAS_500  \
0  125204          1.0         1.0  1.0  2005.0        3.0     1.0       S   
1  125204          1.0         2.0  1.0  2005.0        3.0     1.0       S   
2  125204          1.0         3.0  0.0  2005.0        3.0     1.0       S   
3  125204          1.0         4.0  0.0  2005.0        3.0     1.0       S   

   AGLOMERADO  PONDERA  ...  DECCFR  IDECCFR  RDECCFR  GDECCFR  PDECCFR  \
0        32.0   1542.0  ...      07                06       06            
1        32.0   1542.0  ...      07                06       06            
2        32.0   1542.0  ...      07                06       06            
3        32.0   1542.0  ...      07                06       06            

   ADECCFR  PJ1_1  PJ2_1  PJ3_1  IDIMPP  
0       04    0.0    0.0    0.0   00000  
1       04    0.0    0.0    0.0   10000  
2       04    0.0    0.0    0.0   00000  
3       04    0.0    0.0    0.0   00000  

[4 rows x 176 columns]

### Merger de los dataframes

In [52]:
# Queremos saber cuáles son las variables en cada dataframe y si matchean con las mostradas en las bibliotecas .pdf.

# Variables en el dataframe de 2005:

df_2005_GBA_VAR = df_2005_GBA.columns.tolist()

print(df_2005_GBA_VAR)

['CODUSU', 'NRO_HOGAR', 'COMPONENTE', 'H15', 'ANO4', 'TRIMESTRE', 'REGION', 'MAS_500', 'AGLOMERADO', 'PONDERA', 'CH03', 'CH04', 'CH06', 'CH07', 'CH08', 'CH09', 'CH10', 'CH11', 'CH12', 'CH13', 'CH14', 'CH15', 'CH15_COD', 'CH16', 'CH16_COD', 'NIVEL_ED', 'ESTADO', 'CAT_OCUP', 'CAT_INAC', 'PP02C1', 'PP02C2', 'PP02C3', 'PP02C4', 'PP02C5', 'PP02C6', 'PP02C7', 'PP02C8', 'PP02E', 'PP02H', 'PP02I', 'PP03C', 'PP03D', 'PP3E_TOT', 'PP3F_TOT', 'PP03G', 'PP03H', 'PP03I', 'PP03J', 'INTENSI', 'PP04A', 'PP04B_COD', 'PP04B1', 'PP04B2', 'PP04B3_MES', 'PP04B3_ANO', 'PP04B3_DIA', 'PP04C', 'PP04C99', 'PP04D_COD', 'PP04G', 'PP05B2_MES', 'PP05B2_ANO', 'PP05B2_DIA', 'PP05C_1', 'PP05C_2', 'PP05C_3', 'PP05E', 'PP05F', 'PP05H', 'PP06A', 'PP06C', 'PP06D', 'PP06E', 'PP06H', 'PP07A', 'PP07C', 'PP07D', 'PP07E', 'PP07F1', 'PP07F2', 'PP07F3', 'PP07F4', 'PP07F5', 'PP07G1', 'PP07G2', 'PP07G3', 'PP07G4', 'PP07G_59', 'PP07H', 'PP07I', 'PP07J', 'PP07K', 'PP08D1', 'PP08D4', 'PP08F1', 'PP08F2', 'PP08J1', 'PP08J2', 'PP08J3', '

In [48]:
# Variables en el dataframe de 2025:

df_2025_GBA_VAR = df_2025_GBA.columns.tolist()

print(df_2025_GBA_VAR)

['CODUSU', 'ANO4', 'TRIMESTRE', 'NRO_HOGAR', 'COMPONENTE', 'H15', 'REGION', 'MAS_500', 'AGLOMERADO', 'PONDERA', 'CH03', 'CH04', 'CH05', 'CH06', 'CH07', 'CH08', 'CH09', 'CH10', 'CH11', 'CH12', 'CH13', 'CH14', 'CH15', 'CH15_COD', 'CH16', 'CH16_COD', 'NIVEL_ED', 'ESTADO', 'CAT_OCUP', 'CAT_INAC', 'IMPUTA', 'PP02C1', 'PP02C2', 'PP02C3', 'PP02C4', 'PP02C5', 'PP02C6', 'PP02C7', 'PP02C8', 'PP02E', 'PP02H', 'PP02I', 'PP03C', 'PP03D', 'PP3E_TOT', 'PP3F_TOT', 'PP03G', 'PP03H', 'PP03I', 'PP03J', 'INTENSI', 'PP04A', 'PP04B_COD', 'PP04B1', 'PP04B2', 'PP04B3_MES', 'PP04B3_ANO', 'PP04B3_DIA', 'PP04C', 'PP04C99', 'PP04D_COD', 'PP04G', 'PP05B2_MES', 'PP05B2_ANO', 'PP05B2_DIA', 'PP05C_1', 'PP05C_2', 'PP05C_3', 'PP05E', 'PP05F', 'PP05H', 'PP06A', 'PP06C', 'PP06D', 'PP06E', 'PP06H', 'PP07A', 'PP07C', 'PP07D', 'PP07E', 'PP07F1', 'PP07F2', 'PP07F3', 'PP07F4', 'PP07F5', 'PP07G1', 'PP07G2', 'PP07G3', 'PP07G4', 'PP07G_59', 'PP07H', 'PP07I', 'PP07J', 'PP07K', 'PP08D1', 'PP08D4', 'PP08F1', 'PP08F2', 'PP08J1', 'PP

# Ahora, viendo todas las variables que hay, seleccionamos las 15 que queremos de cada una

# Las variables que elegimos son: 

Variables elegidas:

variables = ['ANO4','PONDERA','CH04','CH06','CH07','CH08','NIVEL_ED',
             'ESTADO','CAT_INAC','IPCF','CAT_OCUP','PP07H','P21','PP04A','PP04C'] # poner esto lindo